# Tarefa investigativa: erros em métodos de interpolação

### Aplicação: reconstrução da temperatura em uma barra aquecida

Uma barra homogênea de comprimento $L=6$ m está sujeita a temperaturas prescritas nas extremidades e recebe calor de uma fonte distribuída em seu interior. Sensores medem a temperatura em posições discretas. A equipe precisa estimar a temperatura em posições onde não há sensor e avaliar a incerteza dessas estimativas.

Esta atividade combina um modelo simplificado de transferência de calor com interpolação numérica. O objetivo não é resolver a equação diferencial de calor por diferenças finitas. O perfil analítico será usado como referência, para que os erros de diferentes métodos de interpolação possam ser medidos e comparados.

Você vai comparar **polinômio global de Lagrange** e uma **spline cúbica**, primeiro com dados exatos e depois com medições perturbadas por ruído.


## 1. Problema físico e equação de transferência de calor

### 1.1 Modelo físico

Considere uma barra sólida homogênea, com comprimento $L=6$ m e seção transversal constante. A temperatura varia principalmente ao longo do eixo $x$. As extremidades são mantidas a temperaturas conhecidas. No interior, uma fonte térmica volumétrica distribui energia ao longo da barra.

A equação geral de energia térmica, para propriedades constantes e sem mudança de fase, pode ser escrita como

$$\rho c_p\frac{\partial T}{\partial t}=k\nabla^2T+\dot q^{\prime\prime\prime}-\rho c_p\,\mathbf{u}\cdot\nabla T,$$

em que $T$ é a temperatura (K ou °C para diferenças), $t$ é o tempo (s), $\rho$ é a massa específica (kg m$^{-3}$), $c_p$ é o calor específico (J kg$^{-1}$ K$^{-1}$), $k$ é a condutividade térmica (W m$^{-1}$ K$^{-1}$), $\dot q^{\prime\prime\prime}$ é a geração volumétrica de calor (W m$^{-3}$) e $\mathbf{u}$ representa o movimento de um fluido, quando há advecção.

Para esta atividade, adotamos uma **barra sólida em regime estacionário**, sem escoamento interno, com condução unidimensional e geração de calor. Assim, $\partial T/\partial t=0$, $\mathbf{u}=0$ e $\nabla^2T=d^2T/dx^2$. A equação fica

$$k\frac{d^2T}{dx^2}+\dot q^{\prime\prime\prime}(x)=0,\qquad 0<x<L,$$

com condições de contorno de Dirichlet

$$T(0)=18\ ^\circ\mathrm C,\qquad T(L)=20.4\ ^\circ\mathrm C.$$

O primeiro termo representa a condução: diferenças espaciais de temperatura geram fluxo de calor. O termo $\dot q^{\prime\prime\prime}(x)$ representa energia adicionada por unidade de volume e de tempo. O sinal positivo de $\dot q^{\prime\prime\prime}$ indica geração interna de calor.

### 1.2 Perfil de referência usado no experimento

Para criar um caso com solução conhecida, definimos $\xi=x/L$ e adotamos

$$T(x)=18+2.4\,\xi+2.5\sin(\pi\xi),\qquad \xi=\frac{x}{L},\quad 0\le x\le L.$$

A parcela linear garante as temperaturas prescritas nas extremidades. A parcela senoidal representa um aquecimento distribuído mais intenso no interior e nulo nas bordas. Derivando duas vezes em relação à coordenada física $x$,

$$\frac{d^2T}{dx^2}=-\frac{2.5\pi^2}{L^2}\sin\left(\frac{\pi x}{L}\right).$$

Portanto, esse perfil satisfaz a equação estacionária se a geração volumétrica for

$$\dot q^{\prime\prime\prime}(x)=\frac{2.5\,k\pi^2}{L^2}\sin\left(\frac{\pi x}{L}\right).$$

Como $\sin(\pi x/L)\ge0$ dentro da barra, a fonte é não negativa. Não precisamos escolher um valor numérico para $k$: o objetivo é verificar a coerência do perfil e estudar a interpolação. A função analítica é uma referência idealizada; em uma aplicação experimental, seria preciso estimar propriedades, fontes, perdas e condições de contorno.

### 1.3 Questão investigativa

**Como a resposta depende da quantidade e posição dos sensores e do ruído de medição?**



### O que significa erro de interpolação?

Em um ponto de avaliação $x$, o erro assinado é $e(x)=p(x)-T(x)$. O erro absoluto é $|e(x)|$. Como a interpolação reproduz os nós, o erro nesses pontos pode ser praticamente zero; por isso, avaliaremos também muitos pontos **entre** os nós.

Usaremos: erro máximo absoluto $E_{max}=\max_j|e_j|$; erro médio absoluto (MAE) $=\frac{1}{m}\sum_j|e_j|$; raiz do erro quadrático médio (RMSE) $=\sqrt{\frac{1}{m}\sum_j e_j^2}$. O RMSE penaliza mais erros grandes.

## 2. Figura do problema e preparação do ambiente

A figura esquematiza a barra, os sensores, as temperaturas nas extremidades e a fonte térmica interna. A curva corresponde ao perfil de referência usado no experimento. Os pontos são as posições onde os sensores amostram a temperatura.

In [ ]:
# Preparação do modelo e bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import BarycentricInterpolator, interp1d, CubicSpline
from matplotlib.patches import Rectangle

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (9, 5)
L = 6.0  # comprimento da barra, m

def temperatura_real(x):
    """Perfil analítico de referência em °C; x é dado em metros."""
    x = np.asarray(x, dtype=float)
    xi = x/L
    return 18 + 2.4*xi + 2.5*np.sin(np.pi*xi)

x_min, x_max = 0.0, L
x_fino = np.linspace(x_min, x_max, 2001)
T_fino = temperatura_real(x_fino)

# Figura do problema: esquema físico e perfil analítico em painéis separados

x_sensores_fig = np.linspace(0, L, 7)
T_sensores_fig = temperatura_real(x_sensores_fig)

fig = plt.figure(figsize=(13, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[0.85, 1.5])
ax_esq = fig.add_subplot(gs[0])
ax_perfil = fig.add_subplot(gs[1])

# Painel (a): barra física, sensores e fonte interna distribuída
ax_esq.set_title("(a) Modelo físico: barra sólida com geração interna de calor", loc="left", fontsize=12, weight="bold")
barra_y, barra_h = 0.40, 0.25
ax_esq.add_patch(Rectangle((0, barra_y), L, barra_h, facecolor="#e8edf2",
                           edgecolor="#34495e", linewidth=1.6, zorder=1))
# Sensores ao longo do eixo da barra
ax_esq.scatter(x_sensores_fig, np.full_like(x_sensores_fig, barra_y+barra_h/2),
               s=65, marker="o", facecolor="#1769aa", edgecolor="white", linewidth=0.9,
               zorder=4, label="Sensores")
# Aquecimento interno: setas iguais identificam geração distribuída, sem sugerir fluxo de calor externo
for xpos in np.linspace(0.45, 5.55, 11):
    ax_esq.annotate("", xy=(xpos, barra_y-0.015), xytext=(xpos, 0.08),
                    arrowprops={"arrowstyle":"-|>", "color":"#d97706", "lw":1.5}, zorder=3)
ax_esq.text(L/2, 0.015, r"Geração volumétrica distribuída $\dot{q}_{\mathrm{gen}}(x)$",
            ha="center", va="bottom", color="#9a4d00", fontsize=10,
            bbox={"facecolor":"white", "edgecolor":"none", "alpha":0.88, "pad":2.5})
ax_esq.text(0, 0.77, r"$T(0)=18{,}0\ ^\circ\mathrm{C}$", ha="left", va="bottom", fontsize=10)
ax_esq.text(L, 0.77, r"$T(L)=20{,}4\ ^\circ\mathrm{C}$", ha="right", va="bottom", fontsize=10)
# Linha de dimensão e eixo longitudinal
ax_esq.annotate("", xy=(L, 1.00), xytext=(0, 1.00),
                arrowprops={"arrowstyle":"<->", "color":"#34495e", "lw":1.2})
ax_esq.text(L/2, 1.025, r"Comprimento $L=6\,\mathrm{m}$", ha="center", va="bottom", fontsize=9)
for i, xpos in enumerate(x_sensores_fig):
    ax_esq.text(xpos, barra_y+barra_h+0.04, rf"$x_{i}$", ha="center", va="bottom", fontsize=8, color="#174a73")
ax_esq.set_xlim(-0.15, L+0.15); ax_esq.set_ylim(0, 1.2)
ax_esq.set_xlabel("Coordenada ao longo da barra, x (m)")
ax_esq.set_yticks([]); ax_esq.grid(False)
for spine in ["left", "right", "top"]: ax_esq.spines[spine].set_visible(False)

# Painel (b): curva de referência e posições amostradas
ax_perfil.set_title("(b) Perfil de temperatura usado como referência para avaliar a interpolação",
                    loc="left", fontsize=12, weight="bold")
ax_perfil.plot(x_fino, T_fino, color="#b23a48", linewidth=2.5,
               label=r"Perfil analítico $T(x)$", zorder=2)
ax_perfil.scatter(x_sensores_fig, T_sensores_fig, s=62, marker="o",
                  facecolor="#1769aa", edgecolor="white", linewidth=0.9,
                  label="Leituras nos sensores (caso sem ruído)", zorder=4)
ax_perfil.fill_between(x_fino, T_fino, np.min(T_fino)-0.3,
                       color="#b23a48", alpha=0.08, zorder=1)
ax_perfil.annotate(r"$T(0)=18{,}0\ ^\circ\mathrm{C}$",
                   xy=(0, temperatura_real(0)), xytext=(0.35, 18.35),
                   arrowprops={"arrowstyle":"->", "color":"#555"}, fontsize=9)
ax_perfil.annotate(r"$T(L)=20{,}4\ ^\circ\mathrm{C}$",
                   xy=(L, temperatura_real(L)), xytext=(4.6, 20.75),
                   arrowprops={"arrowstyle":"->", "color":"#555"}, fontsize=9)
ax_perfil.set_xlim(0, L); ax_perfil.set_ylim(17.7, 22.5)
ax_perfil.set_xlabel("Posição x (m)")
ax_perfil.set_ylabel("Temperatura T(x) (°C)")
ax_perfil.grid(True, alpha=0.25); ax_perfil.legend(loc="lower center", ncol=2, frameon=True)
fig.suptitle("Problema de transferência de calor e reconstrução por interpolação", fontsize=14, weight="bold")
plt.show()

print(f"Temperaturas nas extremidades: T(0)={temperatura_real(0):.1f} °C e T(L)={temperatura_real(L):.1f} °C")

: 

Spline quadrática

In [ ]:
def spline_quadratica_coef(x, y):
    """Spline quadrática C1 com condição c_0=0; retorna a,b,c locais."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)-1
    if n < 1 or len(y) != len(x) or np.any(np.diff(x) <= 0):
        raise ValueError("x deve ser crescente e ter o mesmo tamanho de y.")
    A = np.zeros((3*n, 3*n)); rhs = np.zeros(3*n); row = 0
    # S_i(x_i)=y_i
    for i in range(n):
        A[row, i] = 1; rhs[row] = y[i]; row += 1
    # S_i(x_(i+1))=y_(i+1)
    for i in range(n):
        h = x[i+1]-x[i]
        A[row, i] = 1; A[row, n+i] = h; A[row, 2*n+i] = h*h
        rhs[row] = y[i+1]; row += 1
    # Continuidade da primeira derivada nos nós internos
    for i in range(n-1):
        h = x[i+1]-x[i]
        A[row, n+i] = 1; A[row, 2*n+i] = 2*h; A[row, n+i+1] = -1
        row += 1
    # Condição de contorno: c_0=0
    A[row, 2*n] = 1
    coef = np.linalg.solve(A, rhs)
    return coef[:n], coef[n:2*n], coef[2*n:]

def spline_quadratica_avalia(x, coef, xq):
    a, b, c = coef
    xq = np.asarray(xq, dtype=float)
    idx = np.clip(np.searchsorted(x, xq, side="right")-1, 0, len(a)-1)
    t = xq-x[idx]
    return a[idx] + b[idx]*t + c[idx]*t**2

def criar_metodos(x, y):
    """Retorna dicionário: interpoladores callable para quatro métodos."""
    lagrange = BarycentricInterpolator(x, y)
    linear = interp1d(x, y, kind="linear", bounds_error=True)
    quad_coef = spline_quadratica_coef(x, y)
    quad = lambda z: spline_quadratica_avalia(x, quad_coef, z)
    cubica = CubicSpline(x, y, bc_type="natural")
    return {"Lagrange global": lagrange,
            "Spline linear": linear,
            "Spline quadrática": quad,
            "Spline cúbica natural": cubica}

def calcular_metricas(metodos, x_teste, y_verdadeiro):
    linhas = []
    for nome, metodo in metodos.items():
        erro = np.asarray(metodo(x_teste))-np.asarray(y_verdadeiro)
        linhas.append({"Método": nome,
                       "Erro máximo (°C)": np.max(np.abs(erro)),
                       "MAE (°C)": np.mean(np.abs(erro)),
                       "RMSE (°C)": np.sqrt(np.mean(erro**2))})
    return pd.DataFrame(linhas).sort_values("RMSE (°C)").reset_index(drop=True)

print("Funções de interpolação e métricas definidas.")

## 3. Métodos de interpolação e métricas de erro

O interpolador global de Lagrange passa por todos os sensores usando um único polinômio. Usamos a forma baricêntrica, que evita expandir diretamente os coeficientes e costuma ser numericamente mais estável.

As splines constroem polinômios por intervalos. A spline linear conecta pontos por retas. A quadrática deste notebook impõe continuidade da função e da primeira derivada e usa a condição de contorno $c_0=0$ no primeiro intervalo. A spline cúbica natural impõe continuidade até a segunda derivada e segunda derivada nula nas extremidades.

Para quantificar a diferença, calculamos $e(x)=T_{interp}(x)-T_{ref}(x)$. O erro máximo mostra o pior desvio na malha. O MAE dá o desvio absoluto médio. O RMSE penaliza mais os erros de grande magnitude. Todos têm unidade de graus Celsius.

A malha fina serve como conjunto de pontos de avaliação: os valores nela não são novos sensores. Assim, a comparação mede desempenho entre os nós.

> **Resolução:** nas etapas abaixo, todos os cálculos usam o perfil analítico e a malha de 2001 pontos definidos no início. As temperaturas estão em °C; as posições, em metros.

## Etapa 1 — Interpolação com dados exatos
Suponha primeiro que os sensores sejam perfeitos. Coloque sete sensores em posições igualmente espaçadas, incluindo as duas extremidades. Os valores registrados são então exatamente $T(x_i)$.

### Passo a passo
1. Construa o vetor das posições e avalie o perfil de referência nesses pontos.
2. Crie a interpolação com `criar_metodos`.
3. Avalie cada método em `x_fino`, uma malha densa entre 0 e 6 m.
4. Calcule $E_{max}$, MAE e RMSE comparando cada curva com `T_fino`.
5. Faça um gráfico das temperaturas e outro dos erros assinados.




In [ ]:
# ETAPA 1 — interpolação de 7 sensores sem ruído
x_nos_7 = np.linspace(x_min, x_max, 7)
y_nos_7 = temperatura_real(x_nos_7)
metodos_7 = criar_metodos(x_nos_7, y_nos_7)

print("Posições e temperaturas exatas dos sensores:")
display(pd.DataFrame({"Posição (m)": x_nos_7,
                      "Temperatura (°C)": y_nos_7}).round(6))

metricas_7 = calcular_metricas(metodos_7, x_fino, T_fino)
print("Métricas de erro — menor RMSE primeiro:")
display(metricas_7.style.format({"Erro máximo (°C)": "{:.8f}",
                                 "MAE (°C)": "{:.8f}",
                                 "RMSE (°C)": "{:.8f}"}))

cores = {"Lagrange global": "#2874a6", "Spline linear": "#d97706",
         "Spline quadrática": "#7d3c98", "Spline cúbica natural": "#169873"}
plt.figure(figsize=(11, 6))
plt.plot(x_fino, T_fino, 'k--', linewidth=2.5, label="Temperatura verdadeira")
for nome, func in metodos_7.items():
    plt.plot(x_fino, func(x_fino), label=nome, color=cores[nome], alpha=0.85)
plt.scatter(x_nos_7, y_nos_7, c="black", edgecolor="white", s=65,
            zorder=8, label="7 sensores exatos")
plt.title("Etapa 1 — temperatura estimada por cada método")
plt.xlabel("Posição x (m)"); plt.ylabel("Temperatura (°C)")
plt.grid(alpha=0.25); plt.legend(ncol=2); plt.tight_layout(); plt.show()


In [ ]:
# ETAPA 1 — erros assinados: método menos temperatura verdadeira
fig, eixos = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, (nome, func) in zip(eixos.flat, metodos_7.items()):
    erros = np.asarray(func(x_fino)) - T_fino
    ax.plot(x_fino, erros, color=cores[nome], linewidth=1.8)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
    ax.scatter(x_nos_7, np.zeros_like(x_nos_7), color="black", s=12)
    ax.set(title=nome, xlabel="Posição x (m)", ylabel="Erro assinado (°C)")
    ax.grid(alpha=0.25)
fig.suptitle("Etapa 1 — erro entre sensores (escalas verticais independentes)", fontsize=13)
plt.tight_layout(); plt.show()


### Resposta da Etapa 1

Com **7 sensores exatos e equiespaçados**, os erros são medidos em 2001 pontos, e não somente nos nós, onde os interpoladores passam pelos dados.

- **Lagrange global:** erro máximo aproximado de **0,0000847 °C** e RMSE de **0,0000327 °C** — o menor RMSE neste caso.
- **Spline cúbica natural:** erro máximo aproximado de **0,000506 °C** e RMSE de **0,000239 °C**. A condição de segunda derivada nula nas extremidades é coerente com o perfil senoidal adotado.
- **Spline quadrática:** erro máximo aproximado de **0,0239 °C** e RMSE de **0,0165 °C**. O resultado depende da condição adicional escolhida ($c_0=0$).
- **Spline linear:** erro máximo aproximado de **0,0823 °C** e RMSE de **0,0440 °C**. Os segmentos retos não reproduzem tão bem a curvatura.

**Conclusão:** para este perfil suave, Lagrange é superior com 7 sensores exatos; essa observação não implica que seja a melhor escolha com qualquer quantidade de sensores ou com ruído.

## Etapa 2 — Número e posição dos sensores

Nesta etapa, investigue somente o polinômio global de Lagrange. A interpolação polinomial global pode oscilar entre nós, sobretudo perto das extremidades, quando se usam muitos nós equiespaçados. A distribuição de Chebyshev concentra nós nas bordas e pode reduzir esse efeito.

Para $N$ nós de Chebyshev-Lobatto em $[a,b]$, use

$$x_k=\frac{a+b}{2}+\frac{b-a}{2}\cos\left(\frac{k\pi}{N-1}\right),\quad k=0,\ldots,N-1.$$

Ordene as posições crescentemente. Compare $N=5,9,13,17$ para nós equiespaçados e de Chebyshev. Para cada caso, avalie o erro em todos os pontos da malha fina e registre o erro máximo e o RMSE.

**Atenção:** uma redução de erro neste perfil particular não prova que o mesmo ocorrerá para qualquer função. Relacione a observação à suavidade e à estrutura do perfil usado.

In [ ]:
# ETAPA 2 — Lagrange: influência da quantidade e da posição dos sensores
def nos_chebyshev_lobatto(a, b, N):
    if N < 2:
        raise ValueError("É necessário N >= 2.")
    k = np.arange(N)
    z = np.cos(k * np.pi / (N - 1))
    return np.sort((a + b) / 2 + (b - a) * z / 2)

linhas_nos = []
for N in [5, 9, 13, 17]:
    configuracoes = {
        "Equiespaçados": np.linspace(x_min, x_max, N),
        "Chebyshev-Lobatto": nos_chebyshev_lobatto(x_min, x_max, N)
    }
    for distribuicao, x_sens in configuracoes.items():
        interpolador = BarycentricInterpolator(x_sens, temperatura_real(x_sens))
        erros = np.asarray(interpolador(x_fino)) - T_fino
        linhas_nos.append({"N": N, "Distribuição": distribuicao,
                           "Erro máximo (°C)": np.max(np.abs(erros)),
                           "RMSE (°C)": np.sqrt(np.mean(erros**2))})

tabela_nos = pd.DataFrame(linhas_nos)
print("Comparação da interpolação global com diferentes distribuições de nós:")
display(tabela_nos.style.format({"Erro máximo (°C)": "{:.6e}",
                                 "RMSE (°C)": "{:.6e}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for distribuicao, dados in tabela_nos.groupby("Distribuição", sort=False):
    axes[0].semilogy(dados["N"], dados["Erro máximo (°C)"], 'o-', label=distribuicao)
    axes[1].semilogy(dados["N"], dados["RMSE (°C)"], 'o-', label=distribuicao)
for ax, rotulo in zip(axes, ["Erro máximo (°C)", "RMSE (°C)"]):
    ax.set(xlabel="Número de sensores N", ylabel=rotulo)
    ax.set_xticks([5, 9, 13, 17]); ax.grid(alpha=0.3, which="both")
    ax.legend()
fig.suptitle("Etapa 2 — efeito do posicionamento de sensores no Lagrange")
plt.tight_layout(); plt.show()


### Resposta da Etapa 2

À medida que $N$ aumenta de 5 a 17, **ambas as distribuições reduzem fortemente o erro**, pois a temperatura de referência é uma função analítica e bastante suave. Neste experimento, **Chebyshev-Lobatto** apresenta erro máximo e RMSE menores do que a distribuição equiespaçada nos quatro valores de $N$ examinados. Por exemplo, em $N=9$, os RMSE são da ordem de $10^{-7}$ °C; em $N=17$, os erros de Chebyshev já ficam próximos da precisão de ponto flutuante.

A concentração de nós de Chebyshev nas extremidades ajuda a limitar a amplificação de erros em interpolação polinomial global. Porém, não é correto concluir que Chebyshev sempre terá menor erro em toda função e em qualquer $N$. Em especial, os erros próximos de $10^{-14}$ °C já são indistinguíveis do limite numérico prático em precisão dupla.

## Etapa 3 — Estudo de caso e decisão técnica

Faça $N$ variar de 4 a 40. Compare erro máximo e RMSE em escala logarítmica para Lagrange equiespaçado, Lagrange com Chebyshev e spline cúbica natural. Registre o tempo de construção e de avaliação. Discuta o custo e a estabilidade numérica. Para o perfil suave escolhido, haverá necessariamente um único método superior para todo $N$?

In [ ]:
# ETAPA 3 — N de 4 a 40: erros e tempos de construção/avaliação
from time import perf_counter

# A mediana de 3 repetições reduz a influência de pequenas flutuações no tempo.
def avaliar_custo(construtor, x_consulta, repeticoes=3):
    tempos_construcao, tempos_avaliacao = [], []
    for _ in range(repeticoes):
        inicio = perf_counter()
        interpolador = construtor()
        tempos_construcao.append((perf_counter() - inicio) * 1000)  # ms
        inicio = perf_counter()
        _ = interpolador(x_consulta)
        tempos_avaliacao.append((perf_counter() - inicio) * 1000)  # ms
    return (float(np.median(tempos_construcao)),
            float(np.median(tempos_avaliacao)))

linhas_custo = []
for N in range(4, 41):
    xe = np.linspace(0, L, N)
    xc = nos_chebyshev_lobatto(0, L, N)
    ye, yc = temperatura_real(xe), temperatura_real(xc)
    construcoes = {
        "Lagrange equiespaçado": lambda xe=xe, ye=ye: BarycentricInterpolator(xe, ye),
        "Lagrange Chebyshev": lambda xc=xc, yc=yc: BarycentricInterpolator(xc, yc),
        "Spline cúbica natural": lambda xe=xe, ye=ye: CubicSpline(xe, ye, bc_type="natural")
    }
    for nome, construtor in construcoes.items():
        t_criar, t_avaliar = avaliar_custo(construtor, x_fino)
        interp = construtor()
        erros = np.asarray(interp(x_fino)) - T_fino
        linhas_custo.append({
            "N": N, "Método": nome,
            "Erro máximo (°C)": float(np.max(np.abs(erros))),
            "RMSE (°C)": float(np.sqrt(np.mean(erros**2))),
            "Construção (ms)": t_criar,
            "Avaliação (ms)": t_avaliar
        })

tabela_custo = pd.DataFrame(linhas_custo)
print("Amostra dos resultados: custos são dependentes da máquina e da execução.")
display(tabela_custo[tabela_custo["N"].isin([4, 5, 7, 9, 13, 17, 25, 33, 40])]
        .style.format({"Erro máximo (°C)": "{:.3e}", "RMSE (°C)": "{:.3e}",
                       "Construção (ms)": "{:.3f}", "Avaliação (ms)": "{:.3f}"}))


In [ ]:
# ETAPA 3 — gráficos logarítmicos dos erros e comparação dos tempos
estilos = {"Lagrange equiespaçado": ("#e67e22", "o"),
           "Lagrange Chebyshev": ("#2874a6", "s"),
           "Spline cúbica natural": ("#16a085", "^")}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))
for nome, dados in tabela_custo.groupby("Método", sort=False):
    cor, marcador = estilos[nome]
    for ax, metrica in zip(axes, ["Erro máximo (°C)", "RMSE (°C)"]):
        ax.semilogy(dados["N"], dados[metrica], color=cor,
                    marker=marcador, markevery=5, label=nome)
for ax, metrica in zip(axes, ["Erro máximo (°C)", "RMSE (°C)"]):
    ax.set(xlabel="Número de sensores N", ylabel=metrica)
    ax.grid(alpha=0.28, which="both"); ax.legend()
fig.suptitle("Etapa 3 — acurácia em escala logarítmica (dados exatos)")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.7))
for nome, dados in tabela_custo.groupby("Método", sort=False):
    cor, marcador = estilos[nome]
    for ax, metrica in zip(axes, ["Construção (ms)", "Avaliação (ms)"]):
        ax.plot(dados["N"], dados[metrica], color=cor,
                marker=marcador, markevery=5, label=nome)
for ax, metrica in zip(axes, ["Construção (ms)", "Avaliação (ms)"]):
    ax.set(xlabel="Número de sensores N", ylabel=metrica)
    ax.grid(alpha=0.25); ax.legend()
fig.suptitle("Etapa 3 — custos observados para construir e avaliar 2001 pontos")
plt.tight_layout(); plt.show()

# Primeiro N que satisfaz uma tolerância de erro máximo fisicamente interpretável.
tol = 1e-3  # 0,001 °C
print(f"Primeiro N com erro máximo < {tol:g} °C:")
for nome, dados in tabela_custo.groupby("Método", sort=False):
    aprovados = dados.loc[dados["Erro máximo (°C)"] < tol, "N"]
    print(f"  {nome}: N = {int(aprovados.iloc[0]) if len(aprovados) else 'nenhum até 40'}")


### Discussão da Etapa 3 — custo e estabilidade

- **Não existe um método necessariamente superior para todo $N$.** Com poucos nós (por exemplo, $N=4$), a spline cúbica natural tem menor RMSE que Lagrange neste perfil; com cerca de 7 a 13 sensores exatos, os polinômios globais podem ser mais precisos. O resultado depende do problema e do critério.
- **Lagrange equiespaçado:** quando $N$ é grande, os pesos baricêntricos e as cancelamentos entre termos tornam a interpolação mais sensível a arredondamento e ruído. Neste teste, após atingir erros muito pequenos, o erro pode voltar a crescer para $N$ alto. Não se deve interpretar isso como falha da teoria da interpolação: a aritmética finita também importa.
- **Lagrange Chebyshev:** a escolha dos nós melhora o condicionamento da interpolação global; para o perfil adotado, os erros frequentemente atingem a escala da precisão numérica.
- **Spline cúbica natural:** apresenta erro decrescente e comportamento local, sem os problemas típicos da extrapolação de polinômios globais de alto grau; tem construção e avaliação eficientes para muitos pontos.
- **Tempos:** os números variam conforme CPU, bibliotecas e carga do Google Colab. A comparação deve priorizar a **tendência** e não diferenças de frações de milissegundo. A spline cúbica é uma alternativa escalável; já o método baricêntrico envolve as contribuições de muitos nós em cada avaliação.

Para decisões práticas, diferenças entre erros da ordem de $10^{-13}$ e $10^{-15}$ °C não são relevantes diante de incertezas reais dos sensores.

## Etapa 4 — Estudo de caso e decisão técnica

Uma equipe precisa estimar a temperatura em posições internas da barra, com sensores ruidosos. Não haverá sensores em todos os pontos e a solução verdadeira será desconhecida em campo. Com base nos experimentos, escreva uma recomendação técnica.

A recomendação deve responder:
- que método escolheria com poucos sensores exatos e por quê;
- como a recomendação muda quando se aumenta o número de sensores;
- o que faria quando os sensores apresentam ruído;
- que cuidados adotaria perto das extremidades;
- que limitações permanecem quando a temperatura verdadeira não é conhecida.

Use pelo menos duas métricas da atividade. Diferencie interpolação dentro do intervalo medido de extrapolação além dele.

### Experimento complementar: ruído de medição

O enunciado pede uma recomendação para sensores ruidosos, mas não fornece dados experimentais nem intensidade de ruído. Para fundamentar a resposta sem inventar medições, vamos **simular** erros independentes, normalmente distribuídos, de média zero e desvio padrão $\sigma=0{,}15\ ^\circ\mathrm C$. As temperaturas dos extremos são mantidas fixas porque o modelo as especifica como condições de contorno. Nos ensaios reais, essa hipótese teria de ser verificada.

Compararemos Lagrange equiespaçado, Lagrange Chebyshev-Lobatto, spline cúbica natural e uma **spline de suavização**, acrescentada como alternativa apropriada a dados ruidosos. A spline de suavização não é interpolante exata: ela permite resíduos nos sensores internos e utiliza pesos altos nas extremidades para respeitar as temperaturas prescritas.

In [ ]:
# ETAPA 4 — uma realização de ruído para visualização
from scipy.interpolate import UnivariateSpline

sigma = 0.15  # desvio padrão do ruído (°C): hipótese de simulação
rng_exemplo = np.random.default_rng(2026)
N_exemplo = 17
x_eq = np.linspace(0, L, N_exemplo)
x_ch = nos_chebyshev_lobatto(0, L, N_exemplo)
ruido = rng_exemplo.normal(0.0, sigma, N_exemplo)
ruido[[0, -1]] = 0.0  # extremos conhecidos exatamente no modelo
medidas_eq = temperatura_real(x_eq) + ruido
medidas_ch = temperatura_real(x_ch) + ruido

# Pesos da spline de suavização (1/sigma nos sensores internos);
# nas bordas, peso elevado força o cumprimento das temperaturas prescritas.
pesos = np.full(N_exemplo, 1.0 / sigma)
pesos[[0, -1]] = 1e6
metodos_ruido_exemplo = {
    "Lagrange equiespaçado": BarycentricInterpolator(x_eq, medidas_eq),
    "Lagrange Chebyshev": BarycentricInterpolator(x_ch, medidas_ch),
    "Spline cúbica natural": CubicSpline(x_eq, medidas_eq, bc_type="natural"),
    "Spline de suavização": UnivariateSpline(x_eq, medidas_eq, w=pesos,
                                             s=N_exemplo - 2, k=3)
}
plt.figure(figsize=(11, 5.6))
plt.plot(x_fino, T_fino, color="black", linestyle="--", lw=2.5, label="Perfil verdadeiro")
for nome, func in metodos_ruido_exemplo.items():
    plt.plot(x_fino, func(x_fino), lw=1.6, alpha=0.85, label=nome)
plt.scatter(x_eq, medidas_eq, marker="o", s=35, color="#333333",
            label="Sensores equiespaçados com ruído", zorder=5)
plt.title("Etapa 4 — exemplo com N=17, ruído simulado de σ=0,15 °C")
plt.xlabel("Posição x (m)"); plt.ylabel("Temperatura (°C)")
plt.grid(alpha=0.25); plt.legend(ncol=2); plt.tight_layout(); plt.show()

print("Métricas para esta realização específica de ruído:")
display(calcular_metricas(metodos_ruido_exemplo, x_fino, T_fino)
        .style.format({"Erro máximo (°C)": "{:.5f}",
                       "MAE (°C)": "{:.5f}", "RMSE (°C)": "{:.5f}"}))


In [ ]:
# ETAPA 4 — repetição de experimentos para não depender de uma única amostra
# Os mesmos erros sorteados são aplicados às leituras em cada distribuição de nós.
# São cenários hipotéticos, e não dados de laboratório.
rng = np.random.default_rng(12345)
repeticoes = 30
linhas_montecarlo = []

for N in [7, 17, 33]:
    xe = np.linspace(0, L, N)
    xc = nos_chebyshev_lobatto(0, L, N)
    for r in range(repeticoes):
        eps = rng.normal(0.0, sigma, N)
        eps[0] = eps[-1] = 0.0
        ye = temperatura_real(xe) + eps
        yc = temperatura_real(xc) + eps
        w = np.full(N, 1.0 / sigma)
        w[[0, -1]] = 1e6
        modelos = {
            "Lagrange equiespaçado": BarycentricInterpolator(xe, ye),
            "Lagrange Chebyshev": BarycentricInterpolator(xc, yc),
            "Spline cúbica natural": CubicSpline(xe, ye, bc_type="natural"),
            "Spline de suavização": UnivariateSpline(xe, ye, w=w, s=N - 2, k=3)
        }
        for nome, interpolador in modelos.items():
            e = np.asarray(interpolador(x_fino)) - T_fino
            linhas_montecarlo.append({
                "N": N, "Repetição": r + 1, "Método": nome,
                "Erro máximo (°C)": np.max(np.abs(e)),
                "RMSE (°C)": np.sqrt(np.mean(e**2))
            })

simulacoes = pd.DataFrame(linhas_montecarlo)
resumo_ruido = (simulacoes.groupby(["N", "Método"], as_index=False)
                .agg(RMSE_medio=("RMSE (°C)", "mean"),
                     Emax_medio=("Erro máximo (°C)", "mean")))
resumo_ruido = resumo_ruido.rename(columns={
    "RMSE_medio": "RMSE médio (°C)",
    "Emax_medio": "Erro máximo médio (°C)"})
print(f"Médias de {repeticoes} simulações independentes por N (σ={sigma} °C):")
display(resumo_ruido.style.format({"RMSE médio (°C)": "{:.5g}",
                                   "Erro máximo médio (°C)": "{:.5g}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for nome, dados in resumo_ruido.groupby("Método", sort=False):
    axes[0].semilogy(dados["N"], dados["RMSE médio (°C)"], 'o-', label=nome)
    axes[1].semilogy(dados["N"], dados["Erro máximo médio (°C)"], 'o-', label=nome)
for ax, label in zip(axes, ["RMSE médio (°C)", "Erro máximo médio (°C)"]):
    ax.set(xlabel="Número de sensores N", ylabel=label)
    ax.set_xticks([7, 17, 33]); ax.grid(alpha=0.28, which="both")
    ax.legend(fontsize=8)
fig.suptitle("Etapa 4 — erro sob ruído simulado, escala logarítmica")
plt.tight_layout(); plt.show()


### Recomendação técnica final — respostas solicitadas

**1. Com poucos sensores exatos.** Para a função de temperatura deste problema e **7 sensores**, eu escolheria o **polinômio de Lagrange**: seu RMSE é aproximadamente $3{,}27\times10^{-5}$ °C e seu erro máximo é $8{,}47\times10^{-5}$ °C, menores que os da spline cúbica natural (RMSE $\approx2{,}39\times10^{-4}$ °C; erro máximo $\approx5{,}06\times10^{-4}$ °C). Contudo, com apenas **4 sensores**, a spline cúbica natural é mais precisa no experimento. Portanto, não existe uma resposta única para todo conjunto pequeno de dados.

**2. Quando se aumenta o número de sensores.** Eu evitaria aumentar indiscriminadamente o grau de Lagrange **com nós equiespaçados**. Mesmo com dados exatos, o condicionamento da interpolação global pode levar à amplificação de erros de arredondamento para $N$ elevado. Se for possível escolher a posição dos sensores e os dados forem suficientemente exatos, **Lagrange com Chebyshev-Lobatto** tende a ser uma boa opção aqui. Para muitos sensores distribuídos regularmente, a **spline cúbica natural** oferece comportamento mais local e estável. Compararia RMSE, erro máximo e custo de avaliação antes da escolha.

**3. Com sensores ruidosos.** Polinômios interpoladores e splines cúbicas interpolantes **passam por todas as leituras**, inclusive pelos seus erros de medição. O teste com ruído gaussiano de desvio padrão de **0,15 °C** mostra que o Lagrange equiespaçado com muitos nós pode amplificar fortemente o ruído. Eu usaria uma **spline de suavização** (ou regressão com regularização), escolhendo o parâmetro de suavização com base na incerteza dos sensores ou em validação cruzada. A spline de suavização incluída aqui é **um método adicional** às interpolações estudadas, não uma interpolação exata. As médias dos ensaios da tabela acima ajudam a sustentar essa decisão, mas dependem das hipóteses sobre o ruído.

**4. Cuidados perto das extremidades.** Eu incluiria sensores próximos às bordas, verificaria as condições de contorno $T(0)=18$ °C e $T(6)=20{,}4$ °C e inspecionaria os erros especialmente em regiões com sensores esparsos. Os nós de Chebyshev concentram medições nas bordas e ajudam a controlar a instabilidade do polinômio global. Uma spline **natural** supõe $T''(0)=T''(6)=0$; essa hipótese coincide com o perfil analítico deste exercício, mas pode não ser adequada em outro sistema térmico.

**5. Quando a temperatura verdadeira é desconhecida.** Não é possível calcular o **erro verdadeiro**, RMSE ou erro máximo de referência como fizemos na simulação. É preciso recorrer a sensores de validação independentes, calibração e estimativa da incerteza das medições, além de verificar consistência física e sensibilidade ao posicionamento dos sensores. Erros pequenos nos próprios nós **não provam** boa precisão entre eles.

**Interpolação versus extrapolação.** *Interpolação* estima temperaturas **dentro** do intervalo coberto pelos sensores (neste caso, $0\le x\le6$ m). *Extrapolação* estima **fora** desse intervalo, como em $x=6{,}5$ m, sem suporte direto dos dados: o erro pode crescer rapidamente, sobretudo com polinômios globais de grau elevado. Evitaria extrapolar sem um modelo físico validado.

**Síntese:** com 7 sensores exatos, Lagrange foi o mais preciso neste perfil; para muitos sensores, prefiro uma spline local ou um Lagrange bem condicionado com nós de Chebyshev; quando há ruído real, a prioridade passa a ser **suavização, validação e quantificação de incertezas**, e não apenas passar exatamente pelos pontos medidos.